In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from beta_evidence_field import (OccupancyField,
                                 OccupancyObservation)

from tools.plotting import configure_plots

configure_plots()

# Sampling-Density Normalisation

Range sensors may observe the same region at different sampling densities. Treating every measurement as independent can therefore cause accumulated evidence to depend on the acquisition rate rather than the sensor-state coverage represented by the observations.

To reduce this dependence, each observation is weighted by the inverse local density of observations in sensor-state space,

$$
w_i =
\left[
\sum_j
\exp\left(
-\frac{1}{2} d_s^2(s_i,s_j)
\right)
\right]^{-1}.
$$

The following experiments examine whether the resulting evidence converges as the same underlying observations are sampled increasingly densely.

## Repeated Observations

Consider $N$ observations with identical sensor states. Their pairwise state distances are zero, giving unit similarity between every pair.

Each observation therefore receives weight

$$
w_i = \frac{1}{N},
$$

and the total effective observation weight is

$$
\sum_{i=1}^{N} w_i = 1.
$$

Repeated sampling of an identical sensor state therefore represents a single effective observation, independent of the number of samples.

In [ ]:
sample_counts = np.array([1, 2, 5, 10, 20, 50, 100])


def repeated_observations(n):
    
    return [OccupancyObservation(origin    = np.array([0.0, 0.0]),
                                 direction = np.array([1.0, 0.0]),
                                 range     = 3.0,
                                 returned  = True,
                                 time      = 0.0)
            for _ in range(n)]

In [ ]:
effective_counts = []

for n in sample_counts:

    field = OccupancyField.from_yaml("../config/example_2d_config.yaml")

    observations = repeated_observations(n)
    field.add(observations)

    weights = field._redundancy_weights(field.observations)

    effective_counts.append(np.sum(weights))

effective_counts = np.asarray(effective_counts)

In [ ]:
fig, ax = plt.subplots()

ax.plot(sample_counts,
        sample_counts,
        "o-",
        label = "Unnormalised")

ax.plot(sample_counts,
        effective_counts,
        "o-",
        label = "Sampling-density normalised")

ax.set_xlabel("Number of observations")
ax.set_ylabel("Effective observation count")

ax.legend()

plt.show()

For $N$ identical observations, every observation has a local state-space density of $N$ and therefore receives weight $1/N$.

The unnormalised effective observation count increases directly with the number of samples, whereas the sampling-density-normalised count remains equal to one.

Exact duplication therefore does not increase the effective evidence represented by the observation set.

## Continuous Sampling

Exact repetition represents a limiting case. In practice, a moving sensor acquires observations from continuously changing sensor states.

The same continuous sensor trajectory is now sampled at increasing densities while keeping the trajectory, environment and measurement geometry fixed. This isolates the effect of spatial sampling density on the resulting evidence field.

All observations are assigned the same acquisition time so that temporal separation does not influence the state similarity in this experiment.

In [ ]:
def trajectory(n):

    y = np.linspace(-2.0, 2.0, n)

    return [OccupancyObservation(origin    = np.array([0.0, y_i]),
                                 direction = np.array([1.0, 0.0]),
                                 range     = 3.0,
                                 returned  = True,
                                 time      = 0.0)
            for y_i in y]

In [ ]:
trajectory_counts = [5, 10, 20, 50]

fig, axes = plt.subplots(1,
                         len(trajectory_counts),
                         figsize = (12, 3),
                         sharex = True,
                         sharey = True)

for ax, n in zip(axes, trajectory_counts):

    observations = trajectory(n)

    origins   = np.asarray([observation.origin
                            for observation in observations])

    endpoints = np.asarray([observation.endpoint
                            for observation in observations])

    for origin, endpoint in zip(origins, endpoints):
        ax.plot([origin[0], endpoint[0]],
                [origin[1], endpoint[1]],
                color     = "C0",
                linewidth = 0.8,
                alpha     = 0.6)

    ax.scatter(origins[:, 0],
               origins[:, 1],
               s     = 12,
               color = "C0")

    ax.scatter(endpoints[:, 0],
               endpoints[:, 1],
               s     = 12,
               color = "C1")

    ax.set_title(f"$N = {n}$")
    ax.set_xlabel("$x$")

axes[0].set_ylabel("$y$")

plt.show()

## Effective State-Space Coverage

For a continuously changing trajectory, increasing the sample count introduces genuinely new sensor states as well as increasingly redundant intermediate states.

The total effective observation weight is

$$
N_{\mathrm{eff}}
=
\sum_i w_i.
$$

Unlike the exact-repetition case, this quantity need not equal one. It should initially increase as the trajectory becomes better represented, then approach a finite value once the relevant sensor-state space is sampled sufficiently densely.

In [ ]:
sample_counts = np.array([5, 10, 20, 50, 100, 200, 500])

effective_counts = []

for n in sample_counts:

    field = OccupancyField.from_yaml("../config/example_2d_config.yaml")

    observations = trajectory(n)

    field.add(observations)

    weights = field._redundancy_weights(field.observations)

    effective_counts.append(np.sum(weights))


effective_counts = np.asarray(effective_counts)

In [ ]:
fig, ax = plt.subplots()

ax.plot(sample_counts,
        sample_counts,
        "o-",
        label = "Unnormalised")

ax.plot(sample_counts,
        effective_counts,
        "o-",
        label = "Sampling-density normalised")

ax.set_xlabel("Number of trajectory samples")
ax.set_ylabel("Effective observation count")

ax.legend()

plt.show()

## Pointwise Evidence Convergence

At a fixed query location $x$, the accumulated evidence from $N$ observations can be written as

$$
E_N(x)
=
\sum_{i=1}^{N}
w_i K(x,z_i),
$$

where $K(x,z_i)$ describes the spatial evidence contributed by observation $i$ and $w_i$ accounts for the local density of its sensor state.

Increasing the sampling density introduces additional observations while simultaneously reducing their individual weights. The desired behaviour is therefore convergence to a finite evidence value,

$$
E_N(x) \rightarrow E_{\infty}(x),
$$

rather than evidence increasing in proportion to the number of observations.

In [ ]:
point = np.array([3.0, 0.0])

sample_counts = np.array([5, 10, 20, 50, 100, 200, 500])

occupied_evidence = []
free_evidence     = []

for n in sample_counts:

    field = OccupancyField.from_yaml("../config/example_2d_config.yaml")

    field.add(trajectory(n))

    prior_alpha, prior_beta = field.prior.evaluate(point[None, :])
    query                   = field.query(point)

    occupied_evidence.append(query.alpha - float(np.asarray(prior_alpha)[0]))
    free_evidence.append(query.beta - float(np.asarray(prior_beta)[0]))


occupied_evidence = np.asarray(occupied_evidence)
free_evidence     = np.asarray(free_evidence)

In [ ]:
fig, axes = plt.subplots(1,
                         2,
                         figsize = (10, 4))

axes[0].plot(sample_counts,
             occupied_evidence,
             "o-")

axes[0].set_xlabel("Number of trajectory samples")
axes[0].set_ylabel("Occupied evidence")

axes[1].plot(sample_counts,
             free_evidence,
             "o-")

axes[1].set_xlabel("Number of trajectory samples")
axes[1].set_ylabel("Free-space evidence")

plt.show()

As the trajectory is sampled increasingly densely, the occupied evidence at the selected surface location approaches a stable value.

The number of observations continues to increase, but the local state-space density increases at the same time and reduces the contribution of each individual sample.

The evidence therefore reflects the sensor-state coverage of the trajectory rather than increasing directly with the number of discrete observations.

## Field Convergence

Convergence at a single location does not necessarily imply convergence of the complete field. The experiment is therefore repeated over query locations spanning the observed surface.

As the observation trajectory is sampled increasingly densely, the resulting evidence profiles should approach a common continuous field,

$$
\alpha_N(x) \rightarrow \alpha_{\infty}(x),
\qquad
\beta_N(x) \rightarrow \beta_{\infty}(x).
$$

This tests whether sampling-density normalisation reduces the dependence of the field on the discrete sampling used to represent an otherwise equivalent sensor trajectory.

In [ ]:
y = np.linspace(-3.0, 3.0, 300)

points = np.column_stack((np.full_like(y, 3.0),
                          y))

In [ ]:
profile_counts = [5, 20, 100, 500]

occupied_profiles = []
free_profiles     = []

for n in profile_counts:

    field = OccupancyField.from_yaml("../config/example_2d_config.yaml")

    field.add(trajectory(n))

    prior_alpha, prior_beta = field.prior.evaluate(points)
    query                   = field.query(points)

    occupied_profiles.append(query.alpha - prior_alpha)
    free_profiles.append(query.beta - prior_beta)


occupied_profiles = np.asarray(occupied_profiles)
free_profiles     = np.asarray(free_profiles)

In [ ]:
fig, axes = plt.subplots(1,
                         2,
                         figsize = (10, 4),
                         sharex = True)

for n, occupied, free in zip(profile_counts,
                             occupied_profiles,
                             free_profiles):

    axes[0].plot(y,
                 occupied,
                 label = f"$N = {n}$")

    axes[1].plot(y,
                 free,
                 label = f"$N = {n}$")

axes[0].set_xlabel("Position along surface")
axes[0].set_ylabel("Occupied evidence")

axes[1].set_xlabel("Position along surface")
axes[1].set_ylabel("Free-space evidence")

axes[0].legend()
axes[1].legend()

plt.show()

## Convergence Error

To quantify convergence, each sampled field is compared with a densely sampled numerical reference.

The relative error is defined as

$$
\epsilon_N
=
\frac{
\left\|E_N-E_{\mathrm{ref}}\right\|_2
}{
\left\|E_{\mathrm{ref}}\right\|_2
},
$$

where $E_{\mathrm{ref}}$ is the evidence field produced by the densest sampled trajectory.

Convergence is indicated by

$$
\epsilon_N \rightarrow 0
$$

as the trajectory sampling density increases.

In [ ]:
reference_count = 2000

reference_field = OccupancyField.from_yaml("../config/example_2d_config.yaml")

reference_field.add(trajectory(reference_count))

reference_prior = reference_field.prior.evaluate(points)[0]
reference_query = reference_field.query(points)

reference = reference_query.alpha - reference_prior

In [ ]:
sample_counts = np.array([5, 10, 20, 50, 100, 200, 500, 1000])

errors = []

for n in sample_counts:

    field = OccupancyField.from_yaml("../config/example_2d_config.yaml")

    field.add(trajectory(n))

    prior_alpha = field.prior.evaluate(points)[0]
    query       = field.query(points)

    estimate = query.alpha - prior_alpha

    error = np.linalg.norm(estimate - reference) / np.linalg.norm(reference)

    errors.append(error)


errors = np.asarray(errors)

In [ ]:
fig, ax = plt.subplots()

ax.loglog(sample_counts,
          errors,
          "o-")

ax.set_xlabel("Number of trajectory samples")
ax.set_ylabel("Relative occupied-evidence error")

plt.show()

## Summary

Sampling-density normalisation prevents repeated observations from contributing evidence in direct proportion to their sampling density.

For identical observations, the total effective weight remains constant regardless of the number of samples. For a continuously varying sensor trajectory, increasingly dense sampling produces a convergent evidence field.

The resulting evidence therefore reflects coverage of sensor-state space rather than the arbitrary sampling density used to represent that coverage.